# 01 — Feature Engineering para Series Temporales

En este notebook aprenderemos a construir las features que alimentarán nuestro modelo de predicción de demanda eléctrica.

## Objetivos
- Entender por qué el feature engineering es crítico en series temporales
- Implementar encodings cíclicos para capturar periodicidad
- Crear features de lag y medias móviles
- Evitar el "Training-Serving Skew"
- Implementar el split temporal correcto

In [ ]:
!pip install numpy pandas scikit-learn plotly joblib -q

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Imports OK")

## 1. Generación del dataset para este notebook

In [ ]:
# Dataset sintético para este notebook (1 año, más rápido para Colab)
rng = np.random.default_rng(42)
timestamps = pd.date_range("2023-01-01", "2023-12-31 23:00", freq="h")
n = len(timestamps)

base = 1000.0
trend = 0.003 * np.arange(n)
annual = 80 * np.sin(2 * np.pi * timestamps.dayofyear / 365)
weekly = np.where(timestamps.dayofweek < 5, 60, -80)
hour_arr = timestamps.hour.values
daily = 120 * np.sin(np.pi * (hour_arr - 6) / 12) * (hour_arr >= 6) * (hour_arr <= 22) - 150 * ((hour_arr >= 2) & (hour_arr <= 5))
noise = rng.normal(0, 50, n)
demand = np.maximum(base + trend + annual + weekly + daily + noise, 50)

df = pd.DataFrame({
    "timestamp": timestamps,
    "demand_kwh": np.round(demand, 2),
    "temperature_c": np.round(26 + 6 * np.sin(np.pi * (hour_arr - 6) / 12) + rng.normal(0, 1.5, n), 2),
    "hour": timestamps.hour,
    "day_of_week": timestamps.dayofweek,
    "month": timestamps.month,
    "is_weekend": (timestamps.dayofweek >= 5).astype(int),
})

print(f"Dataset: {len(df):,} registros")
print(df.head())

## 2. Encodings Cíclicos

Las variables temporales como la hora del día tienen una naturaleza **cíclica**: la hora 23 y la hora 0 son adyacentes en el tiempo, pero numéricamente están muy separadas (23 vs 0).

Si usamos el número crudo (`hour = 23` vs `hour = 0`), el modelo creerá que hay una gran diferencia entre medianoche y las 11pm.

**Solución:** Encoding sin/cos — transforma cada variable cíclica en un punto en un círculo unitario:
- `hour_sin = sin(2π * hour / 24)`
- `hour_cos = cos(2π * hour / 24)`

Así, la hora 23 y la hora 0 quedan geométricamente cerca.

In [ ]:
# Sin/Cos encoding para hour, day_of_week, month
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"]  = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"]  = np.cos(2 * np.pi * df["day_of_week"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Visualizar: el encoding cíclico forma un círculo
hours = np.arange(24)
fig = px.scatter(
    x=np.sin(2*np.pi*hours/24),
    y=np.cos(2*np.pi*hours/24),
    text=[str(h) for h in hours],
    title="Encoding cíclico de la hora del día (0-23)",
    labels={"x": "sin(hour)", "y": "cos(hour)"},
)
fig.update_traces(textposition="top center")
fig.show()
print("Nota: La hora 23 y la hora 0 están JUNTAS en el círculo → el modelo las ve como adyacentes ✓")

## 3. Features de Lag

Las series temporales exhiben **autocorrelación temporal**: el valor de demanda de hace 24 horas es un predictor muy fuerte de la demanda actual (mismo patrón diario).

Los **lag features** capturan esta memoria histórica:
- `demand_lag_24h`: demanda de hace exactamente 24 horas (mismo momento ayer)
- `demand_lag_168h`: demanda de hace 168 horas = 7 días (mismo momento la semana pasada)

**Importante:** Los lags introducen `NaN` en las primeras filas, que deberán eliminarse antes del entrenamiento.

In [ ]:
df["demand_lag_24h"]  = df["demand_kwh"].shift(24)
df["demand_lag_168h"] = df["demand_kwh"].shift(168)  # 1 semana

# Visualizar correlación entre demand y demand_lag_24h
sample = df.dropna().head(500)
fig = px.scatter(
    sample, x="demand_lag_24h", y="demand_kwh",
    title="Correlación: demand_kwh vs demand_lag_24h",
    trendline="ols",
    labels={"demand_lag_24h": "Demanda hace 24h", "demand_kwh": "Demanda actual"},
)
fig.show()

corr = df[["demand_kwh","demand_lag_24h","demand_lag_168h"]].corr()
print("Correlaciones con el target:")
print(corr["demand_kwh"])

## 4. Media Móvil

La **media móvil** suaviza la serie y captura la tendencia de corto/mediano plazo. Una ventana de 7 días (168 horas) elimina la variabilidad diaria y deja ver el comportamiento semanal.

In [ ]:
df["demand_rolling_mean_7d"] = df["demand_kwh"].rolling(window=168, min_periods=1).mean().round(2)

# Visualizar
sample = df.iloc[:720].copy()  # primer mes
fig = go.Figure()
fig.add_trace(go.Scatter(x=sample["timestamp"], y=sample["demand_kwh"], name="Demanda real", opacity=0.5))
fig.add_trace(go.Scatter(x=sample["timestamp"], y=sample["demand_rolling_mean_7d"], name="Media móvil 7d", line=dict(width=3)))
fig.update_layout(title="Demanda real vs Media Móvil 7 días", xaxis_title="Fecha", yaxis_title="kWh")
fig.show()

## 5. Features de Interacción

Algunas combinaciones de variables capturan efectos que ninguna variable individual puede explicar. Por ejemplo, el efecto de la temperatura en la demanda varía según la hora del día.

In [ ]:
df["temp_x_hour"] = df["temperature_c"] * df["hour"]
df["is_peak_hour"] = ((df["hour"] >= 18) & (df["hour"] <= 21)).astype(int)

print("Features de interacción creadas:")
print(f"  temp_x_hour: rango [{df['temp_x_hour'].min():.1f}, {df['temp_x_hour'].max():.1f}]")
print(f"  is_peak_hour: {df['is_peak_hour'].mean()*100:.1f}% de las horas son hora pico")

## 6. El problema del Training-Serving Skew

**¿Qué es el Training-Serving Skew?**

Es cuando el modelo en producción recibe datos procesados de forma
DIFERENTE a como fueron procesados en el entrenamiento.

**Ejemplo clásico:**
```
ENTRENAMIENTO:
  scaler = StandardScaler()
  X_train = scaler.fit_transform(X_raw)  # μ=100, σ=20 (del training set)
  model.fit(X_train)

PRODUCCIÓN (INCORRECTO):
  new_scaler = StandardScaler()
  X_prod = new_scaler.fit_transform(X_new)  # ¡μ y σ distintos!
  model.predict(X_prod)  # ← el modelo ve datos en una escala diferente
```

**SOLUCIÓN:**

Guardar el scaler entrenado y reutilizarlo en producción:
```python
joblib.dump(scaler, "scaler.joblib")

# En producción:
scaler = joblib.load("scaler.joblib")
X_prod = scaler.transform(X_new)  # ← usa los mismos μ y σ del training
```

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib, tempfile, os

# --- MAL: crear un scaler nuevo en producción ---
X_train_raw = df[["demand_kwh", "temperature_c"]].dropna().values[:5000]
X_prod_raw  = df[["demand_kwh", "temperature_c"]].dropna().values[5000:5010]

bad_train_scaler = StandardScaler()
X_train_bad = bad_train_scaler.fit_transform(X_train_raw)

bad_prod_scaler = StandardScaler()  # ← scaler DIFERENTE
X_prod_bad = bad_prod_scaler.fit_transform(X_prod_raw)  # ¡INCORRECTO!

# --- BIEN: guardar y reutilizar el mismo scaler ---
good_scaler = StandardScaler()
X_train_good = good_scaler.fit_transform(X_train_raw)

tmp = tempfile.mktemp(suffix=".joblib")
joblib.dump(good_scaler, tmp)  # guardar

loaded_scaler = joblib.load(tmp)  # cargar en producción
X_prod_good = loaded_scaler.transform(X_prod_raw)  # ← CORRECTO (sin fit!)

print("Media del scaler de training:", good_scaler.mean_[:2].round(2))
print("Media del scaler cargado:    ", loaded_scaler.mean_[:2].round(2))
print("Son iguales:", np.allclose(good_scaler.mean_, loaded_scaler.mean_))
os.unlink(tmp)

## 7. Split Temporal

En series temporales **nunca** se hace split aleatorio. Hacerlo introduciría **data leakage**: el modelo vería datos futuros durante el entrenamiento.

El split correcto respeta el orden cronológico:
- **Train**: datos históricos (el modelo aprende)
- **Validation**: periodo siguiente (ajuste de hiperparámetros)
- **Test**: periodo más reciente (evaluación final, se toca solo una vez)

In [ ]:
df_clean = df.dropna(subset=["demand_lag_168h"]).copy()

train = df_clean[df_clean["timestamp"] < "2023-07-01"]
val   = df_clean[(df_clean["timestamp"] >= "2023-07-01") & (df_clean["timestamp"] < "2023-10-01")]
test  = df_clean[df_clean["timestamp"] >= "2023-10-01"]

print(f"Train: {len(train):,} registros ({train['timestamp'].min().date()} → {train['timestamp'].max().date()})")
print(f"Val:   {len(val):,} registros ({val['timestamp'].min().date()} → {val['timestamp'].max().date()})")
print(f"Test:  {len(test):,} registros ({test['timestamp'].min().date()} → {test['timestamp'].max().date()})")

# Visualizar los splits
fig = go.Figure()
for subset, name, color in [(train,"Train","blue"), (val,"Val","orange"), (test,"Test","red")]:
    fig.add_trace(go.Scatter(x=subset["timestamp"], y=subset["demand_kwh"], name=name,
                             line=dict(color=color), opacity=0.7))
fig.update_layout(title="Split Temporal del Dataset", xaxis_title="Fecha", yaxis_title="Demanda (kWh)")
fig.show()

## Resumen: Features Finales

Al final del proceso de feature engineering contamos con las siguientes variables para alimentar el modelo:

In [ ]:
FEATURE_COLS = [
    "temperature_c", "hour", "day_of_week", "month",
    "is_weekend", "demand_lag_24h", "demand_lag_168h",
    "demand_rolling_mean_7d", "hour_sin", "hour_cos",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "temp_x_hour", "is_peak_hour",
]

corr_with_target = df_clean[FEATURE_COLS + ["demand_kwh"]].corr()["demand_kwh"].sort_values(ascending=False)
print("Correlación de features con la demanda (target):")
print(corr_with_target.drop("demand_kwh").to_string())